# Task 1 - Repository cloning and file discovery

The assigned repository is [huggingface/optimum](https://github.com/huggingface/optimum). The ignored
source tree is a shallow clone locked to `a6c775e11118d62712057bd3a8c5649898a5312d`.
Baseline and post-replay line counts are reported separately.

```mermaid
flowchart LR
  U[Assigned public repository] --> C[Shallow clone]
  C --> L[Checkout locked commit]
  L --> R[Count every .py file]
  R --> F[Apply documented exclusions]
  F --> P[Parseability and feature scan]
```

## Reproducible repository scope

The bootstrap script uses a depth-one clone and then checks out the recorded
Optimum commit. Pinning matters more than simply recording the URL: if the
default branch moved, both the discovered-file count and every downstream CPG
count could change. The report therefore keeps the locked baseline separate
from the seven lines added later for the replay demonstration.

Discovery reports two numbers on purpose. The raw count shows every Python file
that exists at the selected commit; the processed count applies the documented
test, setup, build, and generated-file exclusions. This makes the optional
filtering auditable. Vendoring the full third-party tree would make the
submission larger without improving reproducibility, while processing every
generated or test file would increase runtime and obscure the chosen scope.

In [1]:
import json
import subprocess
import sys
from pathlib import Path

root = Path('..').resolve()
sys.path.insert(0, str(root))
from cpg_parser.discovery import discover_repo

evidence = json.loads((root / 'evidence/runtime/verification.json').read_text(encoding='utf-8'))
repo = root / 'source-repo'
if not repo.is_dir():
    repo = root.parent / 'source-repo'
current = discover_repo(repo).as_dict()
is_shallow = subprocess.run(
    ['git', '-C', str(repo), 'rev-parse', '--is-shallow-repository'],
    capture_output=True, text=True, check=True,
).stdout.strip() == 'true'
summary = {
    'locked_baseline': evidence['repository'],
    'current_replay_worktree': {
        key: current[key] for key in (
            'repo_url', 'commit_sha', 'raw_python_files', 'processed_python_files',
            'excluded_python_files', 'total_lines', 'parseable_files',
            'parse_success_rate', 'has_branch', 'has_loop', 'has_call'
        )
    },
    'is_shallow_repository': is_shallow,
    'excluded_files': current['excluded_files'],
}
print(json.dumps(summary, indent=2, ensure_ascii=False))
assert current['commit_sha'] == evidence['repository']['commit_sha']
assert current['processed_python_files'] == evidence['repository']['processed_python_files']
assert is_shallow
print('PASS: shallow clone, locked commit, and discovery counts verified')

{
  "locked_baseline": {
    "repo_id": "huggingface/optimum",
    "url": "https://github.com/huggingface/optimum.git",
    "commit_sha": "a6c775e11118d62712057bd3a8c5649898a5312d",
    "raw_python_files": 74,
    "processed_python_files": 61,
    "baseline_python_lines": 13807,
    "modified_python_lines": 13814,
    "parseable_files": 61,
    "parse_success_rate": 1.0
  },
  "current_replay_worktree": {
    "repo_url": "https://github.com/huggingface/optimum.git",
    "commit_sha": "a6c775e11118d62712057bd3a8c5649898a5312d",
    "raw_python_files": 74,
    "processed_python_files": 61,
    "excluded_python_files": 13,
    "total_lines": 13814,
    "parseable_files": 61,
    "parse_success_rate": 1.0,
    "has_branch": true,
    "has_loop": true,
    "has_call": true
  },
  "is_shallow_repository": true,
  "excluded_files": [
    "setup.py",
    "tests/cli/cli_with_custom_command.py",
    "tests/cli/test_cli.py",
    "tests/common/test_configuration_utils.py",
    "tests/exporters/com

## Reflection

The locked clone produced 74 raw
Python files and 61 files after filtering,
and every processed file parsed successfully. An early version relied on the
current default branch, which made the recorded counts vulnerable to upstream
changes; the replay edit also made a single “total lines” number ambiguous.
Fetching the recorded commit fixed the first problem, and reporting baseline
and modified line counts separately fixed the second. The exclusion list remains
part of the output so that another team can reproduce exactly the same scope.